# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [1]:
#!pip install openai python-dotenv langchain langchain-openai langchain-core
!pip install langchain langchain-openai langchain-core

   ---------------------------------------- 0.0/557.4 kB ? eta -:--:--
   ---------------------------------------- 557.4/557.4 kB 6.3 MB/s  0:00:00
   ---------------------------------------- 0.0/595.1 kB ? eta -:--:--
   ---------------------------------------- 595.1/595.1 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   ---------------------------------------- 874.7/874.7 kB 13.0 MB/s  0:00:00

  Attempting uninstall: websockets

    Found existing installation: websockets 16.0

    Uninstalling websockets-16.0:

      Successfully uninstalled websockets-16.0

   --- ------------------------------------  2/22 [websockets]
   --- ------------------------------------  2/22 [websockets]
   --------- ------------------------------  5/22 [regex]
   ---------- -----------------------------  6/22 [pyyaml]
   -------------- -------------------------  8/22 [orjson]
   --------------------- ------------------ 12/22 [requests-toolbelt]
   ---------

---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [2]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 문자열만 넘겨도 됨. API key를 알아서 찾는다.
out = llm.invoke('LangChain이 뭐야? 한 문장으로.')  # ChatOpenAI는 'invoke'로 바로 질문 가능하다.

In [4]:
out #'AIMessage'를 가져온다.

AIMessage(content='LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86eb24cbce', 'id': 'chatcmpl-DutKpaRYcWTEGYER3b2HGBCqyZs1I', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0262-a759-7240-80f1-6f027c165ebe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
out.content

'LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [6]:
#정석
from langchain_core.messages import HumanMessage # 사람이 보내는 메세지
input_prompt = HumanMessage(content = "Langchain이 뭐야? 한 문장으로.")
message = [input_prompt]  # 이 형태가 정석
out = llm.invoke(message)

In [7]:
out

AIMessage(content='Langchain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 18, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DutSGmk2n9NaXOdOyIEml1ujnDqtS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0269-b0a0-7690-94cc-040fc8c13751-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 33, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
message_cum=""
message_cum.append(message)
message_cum

AttributeError: 'str' object has no attribute 'append'

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


# 기존 OpenAI 방식
# messages = [
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]

messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]


In [10]:
# 기존 OpenAI 방식
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)


answer = llm.invoke(messages)
print(answer.content)
print('타입:', type(answer))

RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 외부 데이터베이스에서 정보를 검색하여 더 정확하고 풍부한 응답을 생성하는 데 사용됩니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


In [11]:
type(answer)

langchain_core.messages.ai.AIMessage

멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [12]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

In [13]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [14]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [15]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\Users\Admin\miniconda3\envs\day4\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
store  #지금은 빈 딕셔너리

{}

In [ ]:
config = {
    "configurable":{
        "session_id":"first"
    }
}
#config는 configurable이라는 키 하나를 갖는 딕셔너리인데, configurable은 또다시 session_id라는 키를 갖는 딕셔너리다.

In [17]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)
# invoke를 메모리객체로 감싸고, config 딕셔너리를 같이 넣어준다.
print(response.content)

안녕하세요, 신범교님! 어떻게 도와드릴까요?


In [18]:
store['first']  # store에 쌓인 내용물 확인

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

In [19]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

In [20]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

# store에서 나온 내용을 AI 쿼리, 그리고 나온 내용을 다시 store에 저장하게된다.

print(response.content)

귀하의 이름은 신범교입니다. 어떻게 도와드릴까요?


In [21]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

In [22]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(conte

In [23]:
config = {"configurable": {"session_id": "second"}}  # 세션 ID 바꿔보자.

In [24]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 당신의 이름은 알 수 없습니다. 하지만 제가 도와드릴 수 있는 다른 질문이나 주제가 있다면 말씀해 주세요!


In [28]:
store['second'].messages

[HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='죄송하지만, 당신의 이름은 알 수 없습니다. 하지만 제가 도와드릴 수 있는 다른 질문이나 주제가 있다면 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 13, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5a1892700', 'id': 'chatcmpl-DutsJCO5Vct6vUZfpz8MGl654oHeC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0282-538c-7770-a9d4-80cdeac77612-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 31, 'total_tokens': 44, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

In [29]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 제가 당신의 이름을 알 수 있는 방법이 없습니다. 당신의 이름이나 다른 정보를 공유하고 싶으시면 그렇게 말씀해 주시면 도움이 될 수 있습니다!


In [30]:
store  #세션id별로 저장된 내용 확인 가능

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [31]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

귀하의 이름은 신범교입니다. 다른 질문이나 요청 사항이 있으신가요?


In [32]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DutmrBKEpWaMjBF9xA5ydHLoW90TS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-2c38-7303-9476-e4fe881b0bb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(conte

In [33]:
# 스트림 방식으로 출력
config = {"configurable": {"session_id": "first"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="") # .stream 만으로 스트림 방식으로 출력이 된다!


신범교라는 이름은 "신(神)"과 "범(凡)" 그리고 "교(敎)"로 나눌 수 있습니다. 

- "신"은 대개 "하늘"이나 "신성한" 것을 의미할 수 있습니다.
- "범"은 일반적으로 "보편적"이라는 뜻으로 해석될 수 있습니다.
- "교"는 "교육"이나 "가르침", 또는 "연결"의 의미를 가지고 있습니다.

따라서 "신범교"라는 이름은 "신성한 교육" 또는 "보편적인 가르침을 가지고 있는 사람" 같은 의미로 해석될 수 있을 것 같습니다. 

물론, 실제 이름의 의미는 부모님의 의도나 가족 전통에 따라 다를 수 있습니다. 이름의 뜻에 대해 더 알고 싶으시면 알려주세요!

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [34]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요, 교수님! 저희 수업 정말 유익했습니다. 교수님께서 설명해 주신 내용을 통해 많은 것들을 배울 수 있었어요. 교수님은 제 지도 교수님이시고, 저희 연구에 많은 도움을 주시는 분이십니다. 교수님, 수업 중 특히 어떤 부분이 마음에 드셨나요?


In [42]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
print(out)
print(out.content)
print(out.response_metadata)

content='안녕하세요, 교수님! 저희 수업 정말 유익했습니다. 교수님께서 설명해 주신 내용을 통해 많은 것들을 배울 수 있었어요. 교수님은 제 지도 교수님이시고, 저희 연구에 많은 도움을 주시는 분이십니다. 교수님, 수업 중 특히 어떤 부분이 마음에 드셨나요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 80, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a849775b18', 'id': 'chatcmpl-DuuAXYcLIMqhoAP5k4JCsoyOYWTy8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f0293-9526-7932-bacc-f29aa066f2db-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 75, 'total_tokens': 155, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
안녕하세요, 

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [44]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()


In [45]:

result = model.invoke(messages) # 1단계 메시지 호출


In [46]:
parser.invoke(result) # 2단계 parser로 str 추출

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 주신 자료와 예시들이 이해하는 데 많은 도움이 되었어요. 교수님은 저희의 지도교수가 되시는 분이십니다! 늘 저희에게 귀한 가르침을 주셔서 감사합니다.'

In [47]:
chain = model | parser
chain.invoke(messages)
#둘다 invoke로 실행하는 함수이므로, lanchain으로 일련의 과정으로 묶을 수 있다.

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 강의하신 내용을 통해이번 주 연구 주제에 대한 통찰을 얻을 수 있었습니다. 교수님은 우리 학과의 주임 교수님이시죠?'

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [48]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [50]:
character_a = '선생님'
f'선생님은 {character_a}이시다' #f word로 변수를 바로 표시되게 할 수 있음

'선생님은 선생님이시다'

In [51]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

In [ ]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
}) #template type으로 invoke할때는 딕셔너리를 넣어줘야한다. 그러니까 invoke가 str type, dict type 둘다 되는것!

In [52]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [53]:
chain = prompt_template | model | parser  #템플릿에 딕셔너리로 특정 변수 입력하기 -> 모델에게 질의하기 -> 답변에서 str 추출하기

In [54]:
chain.invoke({
    "character_a": "동료 친구",
    "character_b": "정한솔",
    "lesson": "수업 분위기"
})  #이렇게 함수 한줄로 반복적으로 물어보는 질의문에 변수를 넣어 답까지 얻을 수 있다.

'안녕! 오늘 수업 분위기는 괜찮았어. 교수님도 잘 설명해주셨고, 학생들도 활발하게 질문하더라. 너는 내 동료 친구니까 수업에 적극적으로 참여했던 것 같은데? 내 이름은 정한솔이야. 너는 누구지? 수업에서 어떤 역할을 맡고 있었는지 궁금해!'

In [ ]:
chain.invoke({
    "character_a": "",
    "character_b": "",
    "lesson": ""
})